# Elie & Theunissen (2019) on CRCNS aa-4

Paper: Elie JE, Theunissen FE (2019). Invariant neural responses for sensory categories revealed by the
time-varying information for communication calls. PLoS Comput Biol 15(9):e1006698,
doi:10.1371/journal.pcbi.1006698.

Dataset: CRCNS aa-4, "Simultaneous extracellular recordings of avian auditory neurons in zebra finches presented
with all the repertoire of vocalizations used by this species for vocal communication", doi:10.6080/K00C4T06,
data description version 0.6 (July 15, 2019). CRCNS data sets are not versioned, so the version is recorded as
`unversioned`; the only published state is the one in `aa-4/live/`, and it is the one read here. The files are
streamed from the public S3 bucket `crcnsarchive`; their MD5 sums in `aa-4/live/checksums.md5`, read on
2026-11-13, are:

| file | md5 |
|---|---|
| data/BlaBro09xxF.tar.gz | ee387cc051e5fa5ee66a5f4db5d8bae6 |
| data/GreBlu9508M.tar.gz | 16f1bb223946288f8b426d6382f06510 |
| data/LblBlu2028M.tar.gz | 80c181eb8bd70ddbbd9b3aa46587dfb5 |
| data/WhiBlu5396M.tar.gz | 2ba7776cd4130d0090b13e5f5c0359fd |
| data/WhiWhi4522M.tar.gz | c47d6660019a2a0d1e3375ce7da68bf1 |
| data/YelBlu6903F.tar.gz | 6027933cdf765dd0b67f811b1a133095 |

Outside input: the tutorial model cell `ModelCell_InfoCumInfo_Tutorial.mat` of the paper's code repository,
github.com/julieelie/PoissonTimeVaryingInfo at commit 978020328321c7ed83be12c0152bed85ec77e218 (claim
fig1-model-neurons, `data: elsewhere`).

Claims file: `claims/elie-2019.yaml`, written from the version of record (PMC6762074) with its S1 to S7 Fig
legends.

| claim | status here |
|---|---|
| units-404 | blocked: the 914 single auditory units and the 404 selected ones cannot be identified in the files |
| units-337, s1-temporal-resolution, fig2a-poisson-test, fig2cd-fano-factor, fig2b-noise-correlations, s2-spike-counts | blocked: rest on the 404 units (variants on all auditory single units shown, not deciding) |
| fig6a-onset-sustained-information, fig6c-new-information, fig6b-fixed-rate-fraction, fig7a-time-constants, fig7b-k300, k300-rate-regression, fig9a-mean-cii, fig9a-top-quartile, s7-invariant-information, fig9b-cii-tau-ratio, fig9c-cii-k300, cii-unimodal | blocked: rest on the 337 units of units-337 |
| fig3-5-example-neurons | blocked: the paper does not name the three units |
| table1-anatomy-counts, fig10-cii-distance, fig10-cii-area-anova, fig10-tau-k300-distance, fig10-no-anatomy-categorical | blocked: no anatomical location in the data set |
| fig1-model-neurons | tested, on the one model neuron the paper's code repository holds |

## Loading

Each bird's tar.gz is streamed and its h5 files are read in memory (`elie.load_all`); only spike times from -600 to 600 ms around stimulus onset, the stimulus attributes and the stored coherence are kept.

In [ ]:
import time, resource, collections
import numpy as np, pandas as pd
from scipy import stats
import elie

t0 = time.time()
U = elie.load_all()
print(f"loaded {len(U)} h5 files in {time.time() - t0:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB (this process)")

## Data check

Counts of files, units, stimuli and trials, and the ranges of the variables used, against the data description and the paper.

In [ ]:
pd.set_option("display.width", 200)
df = pd.DataFrame([{
    "bird": u["bird"], "file": u["file"], "site": u.get("site"), "electrode": u.get("electrode"),
    "sortType": u.get("sortType"), "ldepth": u.get("ldepth"), "rdepth": u.get("rdepth"),
    "spike_sorted_name": "_ss" in u["file"], "groups": "+".join(u["groups"]),
    "iar": elie.iar(u), "n_call_stims": len(elie.call_stims(u)) if elie.is_single(u) else np.nan,
    "n_stims_all": len(u["stims"]), "n_unsorted_trials": u["n_unsorted_trials"],
} for u in U])
df["single"] = [elie.is_single(u) for u in U]
print("h5 files by bird and sortType:")
print(pd.crosstab(df.bird, df.sortType.astype(str), margins=True))
print("\nfiles named as spike sorted (_ss) by sortType:")
print(pd.crosstab(df.spike_sorted_name, df.sortType.astype(str)))
print("\nsites per bird:", df.groupby("bird").site.nunique().to_dict())
print("depths (um), left:", np.nanmin(df.ldepth.astype(float)), "to", np.nanmax(df.ldepth.astype(float)),
      "; right:", np.nanmin(df.rdepth.astype(float)), "to", np.nanmax(df.rdepth.astype(float)))
S = [u for u in U if elie.is_single(u)]
sd = df[df.single]
print(f"\nsingle units: {len(S)}; with I_AR >= 2.3 bits/s: {(sd.iar >= 2.3).sum()}; I_AR range "
      f"{sd.iar.min():.2f} to {sd.iar.max():.2f} bits/s")
print(f"call stimuli per single unit: mean {sd.n_call_stims.mean():.1f}, sd {sd.n_call_stims.std():.1f}, "
      f"min {sd.n_call_stims.min():.0f}, max {sd.n_call_stims.max():.0f}")
print(f"all stimuli per single unit (any type): mean {sd.n_stims_all.mean():.1f}, sd {sd.n_stims_all.std():.1f}, "
      f"min {sd.n_stims_all.min()}, max {sd.n_stims_all.max()}")
ntr = collections.Counter(len(s["trials"]) for u in S for s in elie.call_stims(u))
print("trials per call stimulus:", dict(sorted(ntr.items())))
print("call stimuli by call type (unit x stimulus):",
      dict(collections.Counter(s["ct"] for u in S for s in elie.call_stims(u))))
allsp = np.concatenate([np.concatenate(s["trials"]) for u in S for s in elie.call_stims(u) if sum(map(len, s["trials"]))])
print(f"spike times kept: {allsp.size} spikes, from {allsp.min():.4f} to {allsp.max():.4f} s around onset (unit s)")
dur = np.array([s["stim_duration"] for u in S for s in elie.call_stims(u)], float)
print(f"stim_duration of call stimuli: {dur.min():.3f} to {dur.max():.3f} s, median {np.median(dur):.3f}")
print("trials with stored spike times out of order:", int(df.n_unsorted_trials.sum()))

Checks against the description and the paper, from the output above:

- The data description says 688 multi-units were recorded and 914 single auditory units were saved, one h5 file
  each. The files hold 845 unsorted (`tdt`) multi-unit files and 1453 spike-sorted (`_ss`) files, of which 1281
  are labelled single units (`sortType` "single", or "singl" in three files), 63 multi and 109 noise. The 914
  are not marked. This is recorded as a finding.
- Six birds, as described; WhiBlu5396M is the bird missing from the paper's Table 1. The description's file
  organisation section names it WhiBlu5896M once.
- Call stimuli per single unit: mean 95.4 (63 to 120), against the paper's 114 on average (sd 22, 34 to 123).
  All stimuli of any kind: mean 121.9 (73 to 137), against the description's 144 (64 to 153). The counts do not
  match either source.
- Trials per stimulus: 9 to 12, with 1874 unit-stimulus pairs at 12, against "min=9, max=11".
- Spike times are in seconds from stimulus onset (range -0.6 to 0.6 s as kept); no trial holds spike times out of
  order. `stim_duration` (0.08 to 2.9 s, median 2.5 s) is longer than most calls; it is not used.

No recording is deposited twice: the spike trains of every single unit to its call stimuli are compared.

In [ ]:
import hashlib
keys = collections.defaultdict(list)
for u in S:
    st = sorted(elie.call_stims(u), key=lambda s: s["stim_md5"])
    h = hashlib.md5()
    for s in st:
        for t in s["trials"]:
            h.update(np.asarray(t, np.float32).tobytes()); h.update(b"|")
    keys[h.hexdigest()].append(u["bird"] + "/" + u["file"])
dups = [v for v in keys.values() if len(v) > 1]
print(f"single units with identical spike trains to all call stimuli: {len(dups)} groups")
for v in dups[:10]:
    print("  ", v)

Clock. In 50 ms windows around onset, a Kruskal-Wallis test of the spike counts across the nine call types, per
single unit. Before onset the counts cannot depend on the call that follows; after onset they can.

In [ ]:
# Contrast between conditions around the trial start: in 50 ms windows from -300 to
# 600 ms, a Kruskal-Wallis test of the trial spike counts across the nine call types,
# per unit. Before onset the counts cannot depend on the call type.
wins = np.arange(-0.3, 0.6, 0.05)
frac = []
rate = []
for w0 in wins:
    sig, r = 0, []
    for u in S:
        st = elie.call_stims(u)
        grp = collections.defaultdict(list)
        for s in st:
            for t in s["trials"]:
                n = int(((t >= w0) & (t < w0 + 0.05)).sum())
                grp[s["ct"]].append(n); r.append(n)
        vals = list(grp.values())
        try:
            p = stats.kruskal(*vals).pvalue
        except ValueError:
            p = 1.0
        sig += p < 0.01
    frac.append(sig / len(S)); rate.append(np.mean(r) / 0.05)
clock = pd.DataFrame({"window_start_ms": (wins * 1000).round().astype(int), "fraction_units_p<0.01": np.round(frac, 3),
                      "mean_rate_spikes_per_s": np.round(rate, 2)})
print(clock.to_string(index=False))

Before onset 0.9% to 1.9% of units differ across call types at p < 0.01, near the 1% false-positive rate, and
the mean rate is flat at about 6.1 spikes/s; from the 0 to 50 ms window on, 44% to 66% do and the rate rises to
22 spikes/s. The spike times share the clock of the stimulus onsets, for the population; a check per unit was not
made.

## units-404

> The analysis uses 404 of the 914 isolated single units, those from which call type could be decoded above
> chance in the authors' earlier study.

Criterion: reading (a), a list of the 404 units from the data set's documentation or the 2015 paper; reading (b),
the 2015 decoding selection redone as that paper describes it. Blocked as methods_underspecified when neither
can be carried out.

In [ ]:
n_single = len(S)
n_aud = int((sd.iar >= 2.3).sum())
print(f"files labelled single units (sortType single or singl): {n_single}")
print(f"  of which I_AR >= 2.3 bits/s (the 2015 threshold for auditory spike-sorted units): {n_aud}")
print(f"spike-sorted files (_ss) of any sortType: {int(df.spike_sorted_name.sum())}")
print("expected: 914 single auditory units, of which 404 selected by the 2015 decoding")
print("per bird, single / single with I_AR >= 2.3:")
print(sd.assign(aud=sd.iar >= 2.3).groupby("bird").agg(single=("file", "size"), auditory=("aud", "sum")).to_string())
attrs = sorted({k for u in U for k in u if k not in ("stims", "class_info", "groups")})
print("unit-level fields read from every file:", attrs)
print("groups found in files:", sorted({g for u in U for g in u["groups"]}))

The files hold no list or flag of the 404 units, and the data description and the 2015 paper (Elie &
Theunissen 2015, Eur J Neurosci 41:546-567, read at PMC4347984) give none, so reading (a) cannot be carried out.
Reading (b) starts from the 914 single auditory units, which the 2015 paper defines by an SNR of the spike
snippets above 5 among the spike-sorted units with I_AR of at least 2.3 bits/s. The SNR is not stored in the h5
files, and the snippet files (`.f32`) are undocumented and exist for only some sites. The I_AR threshold alone
leaves 1246 units, not 914. The 2015 decoding would also have to be run on a set that is not the paper's.

**Result: blocked.** The data set holds 1281 files labelled single units, 1246 of them with I_AR of at least 2.3
bits/s; neither the 914 nor the 404 can be identified.

## Cost of the paper's pipeline

Every claim from units-337 on needs the locally adaptive KDE rate of each unit and stimulus, and its 10
jackknife estimates (Methods R), before the Monte Carlo cumulative information. The time of the KDE alone, on a
sample of call stimuli of three auditory single units:

In [ ]:
# Cost of the paper's rate estimate (Methods R) on the data: time the locally
# adaptive KDE on the call stimuli of a few auditory single units, for the full
# set of trials and one jackknife set, and scale to 404 units x 11 estimates per stimulus.
rng_cost = np.random.default_rng(404)
aud = [u for u in S if elie.iar(u) >= 2.3]
sample = [aud[i] for i in rng_cost.choice(len(aud), 3, replace=False)]
n_kde, t_kde = 0, 0.0
for u in sample:
    for s in elie.call_stims(u)[:10]:
        tr = [t[(t >= 0) & (t < 0.6)] * 1000 for t in s["trials"]]
        for drop in (None, 0):
            use = [t for i, t in enumerate(tr) if i != drop]
            t1 = time.time()
            elie.kde_rate(np.concatenate(use), len(use), 600)
            t_kde += time.time() - t1; n_kde += 1
per = t_kde / n_kde
mean_stims = sd[sd.iar >= 2.3].n_call_stims.mean()
est_h = per * 404 * mean_stims * 11 / 3600
print(f"{n_kde} KDE estimates on 3 units: {per:.2f} s each")
print(f"estimate for the rates alone (404 units x {mean_stims:.0f} stimuli x 11 estimates): {est_h:.0f} h on one core, "
      f"{est_h / 4:.0f} h on 4 cores")

The rate estimates alone would take more than 100 hours on one core, and more than 25 hours on the runner's 4
cores, before any Monte Carlo estimate of cumulative information; this is beyond the limit of a notebook even if
the 404 units could be identified. The time per estimate varies between runs; the printed estimate is the one
for this run.

## Variants on all auditory single units (not deciding any outcome)

The spike-count measures of s2-spike-counts, fig2cd-fano-factor and fig2b-noise-correlations, and the coherence
part of s1-temporal-resolution, on all 1246 single units with I_AR of at least 2.3 bits/s. The claims rest on the
404 units; these numbers describe the data set's units and decide nothing.

In [ ]:
# Not deciding any outcome: the spike-count measures of s2-spike-counts,
# fig2cd-fano-factor, fig2b-noise-correlations and the coherence part of
# s1-temporal-resolution, on all auditory single units (I_AR >= 2.3 bits/s),
# since the 404 units cannot be identified.
rows = []
for u in aud:
    cs = elie.counts(u)
    allc = np.concatenate([c.ravel() for c in cs])
    sc1, sc2 = elie.stim_corr(cs)
    rows.append({"bird": u["bird"], "max_count": int(allc.max()), "n8": int((allc == 8).sum()),
                 "n_bins": allc.size, "sum": int(allc.sum()),
                 "mean_trialavg": float(np.mean([c.mean(0).mean() for c in cs])),
                 "fano_a": elie.fano(cs, True), "fano_b": elie.fano(cs, False),
                 "noise_corr": elie.noise_corr(cs), "stim_corr_over_stims": sc1, "stim_corr_within_stim": sc2,
                 "coh_frac_50": elie.coherence_fraction(u)})
V = pd.DataFrame(rows)
print(f"units: {len(V)}")
print(f"s2: largest count in a 10 ms bin {V.max_count.max()}; bins with 8 spikes {V.n8.sum()}; "
      f"bins with more than 8: {(V.max_count > 8).sum()} units have one; mean count per bin over trials "
      f"{V['sum'].sum() / V.n_bins.sum():.3f}, over trial-averaged counts {V.mean_trialavg.mean():.3f} "
      f"(paper, 404 units: 8, 1, 0.108)")
print("  distribution of the largest count per unit:", dict(sorted(collections.Counter(V.max_count).items())))
for col in ["fano_a", "fano_b"]:
    x = V[col].dropna()
    print(f"Fano ({col}): mean {x.mean():.3f}, SEM {x.std() / np.sqrt(len(x)):.3f}, range {x.min():.2f} to {x.max():.2f}; "
          f"t-test vs 1 p = {stats.ttest_1samp(x, 1).pvalue:.2g}; Wilcoxon vs 1 p = {stats.wilcoxon(x - 1).pvalue:.2g} "
          f"(paper: 1.04, SEM 0.04)")
x = V.noise_corr.dropna()
print(f"noise correlation: median {x.median():.3f}, mean {x.mean():.3f}, skewness {stats.skew(x):.2f}, range {x.min():.2f} to {x.max():.2f}")
for col in ["stim_corr_over_stims", "stim_corr_within_stim"]:
    x = V[col].dropna()
    print(f"stimulus correlation ({col}): median {x.median():.3f}, mean {x.mean():.3f}, range {x.min():.2f} to {x.max():.2f} (paper: around 0.4)")
x = V.coh_frac_50 * 100
print(f"coherence information below 50 Hz: {x.mean():.1f}% +- {x.std():.1f}% (mean +- SD); units losing more than 5%: "
      f"{(x < 95).sum()} of {len(x)} (paper, 404 units: 96.7 +- 6.9%, 88 of 404)")
print("\nby bird (medians):")
print(V.groupby("bird")[["fano_a", "noise_corr", "stim_corr_over_stims", "coh_frac_50"]].median().round(3).to_string())

On the 1246 auditory single units: the largest count in a 10 ms bin is 9 (4 units) with 442 bins at 8, and the
mean count per bin is 0.132, against 8, 1 and 0.108 for the paper's 404. The time-averaged Fano factor is 1.027
(SEM 0.002) under the code's reading and 1.054 (SEM 0.003) leaving out empty bins, both significantly above 1 at
this n. The noise correlation has median 0.152 and positive skew (1.09); the stimulus correlation has median 0.291
over stimuli and 0.239 within stimuli. The stored coherence puts 65.8% +- 19.0% of I_AR below 50 Hz, far from the
paper's 96.7% +- 6.9%; the stored curve, from 4 to 496 Hz, may not be the one the paper's S1 Fig used.

## fig1-model-neurons

> For simulated Poisson neurons (rate, onset and temporal codes, 4 stimuli, 10 trials), the KDE recovers the
> model rates with an average error below 0.02 spikes/ms, the cumulative information approaches the 2-bit
> maximum, and the temporal neuron's peak instantaneous information is 0.5 bits.

Criterion: with the tutorial's model cells, mean absolute difference between KDE and model rates below 0.02
spikes/ms, the cumulative information of each model neuron reaching at least 1.8 bits before it ends, and the
temporal neuron's peak instantaneous information within 0.5 +- 0.05 bits.

The repository holds one model cell, a temporal neuron (each stimulus drives a response at a different time).
The rate and onset neurons are not there. The KDE is the Python port of ssvkernel (`adaptivekde`); the bias
correction is the jackknife, N I - (N-1) mean(I_k), with the jackknife set k used for every stimulus (Methods,
Bias correction; the equation is not in the text read, and this is the standard form). The cumulative
information uses 200,000 Monte Carlo samples with the paper's proposal, the product of the per-bin marginals
(Methods, The cumulative information).

In [ ]:
import io, fsspec, scipy.io as sio
REPO = "https://raw.githubusercontent.com/julieelie/PoissonTimeVaryingInfo/978020328321c7ed83be12c0152bed85ec77e218/"
with fsspec.open(REPO + "ModelCell_InfoCumInfo_Tutorial.mat", block_size=2**24).open() as f:
    M = sio.loadmat(io.BytesIO(f.read()))
n_stim, n_tr, dur = int(M["NStims"][0, 0]), int(M["NTrials"][0, 0]), int(M["NR_Duration"][0, 0])
model_bins = M["Input_Spike_count"]            # model mean count per 10 ms bin, stimulus x bin
print(f"tutorial model cell: {n_stim} stimuli, {n_tr} trials, {dur} ms at {int(M['Response_samprate'][0, 0])} Hz, "
      f"{model_bins.shape[1]} bins of {int(M['Step'][0, 0])} ms")
print("model mean count per 10 ms bin, range", model_bins.min(), "to", model_bins.max())
print("start (ms) of the bins where each stimulus drives a response (model count > 0.1):",
      [[int(b) for b in np.flatnonzero(r > 0.1) * 10] for r in model_bins])
arr = [M["Spike_array"][0, 0][f"stim{s + 1}"] for s in range(n_stim)]   # trial x 1 ms
spk = [[np.repeat(np.flatnonzero(a[t]), a[t][a[t] > 0].astype(int)) + 1.0 for t in range(n_tr)] for a in arr]
print("spikes per stimulus (all trials):", [int(a.sum()) for a in arr])

rate = np.array([elie.kde_rate(np.concatenate(sp), n_tr, dur) for sp in spk])           # spikes/ms
jk = np.array([[elie.kde_rate(np.concatenate([x for i, x in enumerate(sp) if i != k]), n_tr - 1, dur)
                for sp in spk] for k in range(n_tr)])                                     # set x stim x ms
saved = np.array([M["Spike_filtered"][0, 0][f"stim{s + 1}"].ravel() for s in range(n_stim)])
print(f"KDE here against the KDE saved by the MATLAB tutorial: mean |diff| {np.abs(rate - saved).mean():.5f}, "
      f"max {np.abs(rate - saved).max():.5f} spikes/ms")
model = np.repeat(model_bins / 10.0, 10, axis=1)                                          # spikes/ms at 1 kHz
mae = np.abs(rate - model).mean(1)
print("mean |KDE - model rate| per stimulus (spikes/ms):", np.round(mae, 4), "; all:", round(float(mae.mean()), 4),
      "; RMS:", round(float(np.sqrt(((rate - model) ** 2).mean())), 4))

mu = np.array([elie.bin_means(r) for r in rate])
mu_jk = np.array([[elie.bin_means(r) for r in s] for s in jk])
mu_model = np.maximum(model_bins, elie.MU_MIN)
N = n_tr

def jackknife(full, reps):
    """Bias-corrected value N*I - (N-1)*mean(I_k) and its jackknife standard error."""
    reps = np.asarray(reps)
    bc = N * full - (N - 1) * reps.mean(0)
    se = np.sqrt((N - 1) / N * ((reps - reps.mean(0)) ** 2).sum(0))
    return bc, se

ii, ii_se = jackknife(elie.inst_info(mu), [elie.inst_info(m) for m in mu_jk])
ii_model = elie.inst_info(mu_model)
SEED_F1 = 1
NS = 200_000
ci_full = elie.cum_info_mc(mu, NS, SEED_F1)
ci, ci_se = jackknife(ci_full, [elie.cum_info_mc(m, NS, SEED_F1) for m in mu_jk])
ci_model = elie.cum_info_mc(mu_model, NS, SEED_F1)
tab = pd.DataFrame({"bin_end_ms": np.arange(1, mu.shape[1] + 1) * 10, "inst_bc": ii.round(3), "inst_se": ii_se.round(3),
                    "inst_model": ii_model.round(3), "cum_bc": ci.round(3), "cum_se": ci_se.round(3),
                    "cum_model": ci_model.round(3)})
print(tab.to_string(index=False))
print(f"peak bias-corrected instantaneous information {ii.max():.3f} bits at bin ending {10 * (ii.argmax() + 1)} ms "
      f"(model rate: {ii_model.max():.3f}); uncorrected peak {elie.inst_info(mu).max():.3f}")
print(f"cumulative information: largest {ci.max():.3f} bits, last bin {ci[-1]:.3f} bits (model rate: {ci_model[-1]:.3f}); "
      f"largest jackknife SE {ci_se.max():.3f} bits; maximum log2(4) = 2 bits")
print("same with the MATLAB tutorial's saved results: peak instantaneous",
      round(float(M["Info_bcorr"].max()), 3), "; last cumulative", round(float(M["Icum_EstMonteCarloOpt_bcorr"][0, -1]), 3))
# Variant: jackknife replicates with independent Monte Carlo draws (seeds 2 to 11).
ci_ind, _ = jackknife(ci_full, [elie.cum_info_mc(m, NS, 2 + k) for k, m in enumerate(mu_jk)])
print(f"variant, independent draws per replicate: largest {ci_ind.max():.3f} bits, last bin {ci_ind[-1]:.3f} bits")

**Result: partially reproduced.** On the tutorial's temporal neuron the KDE differs from the model rate by 0.009
spikes/ms on average (below 0.02), the bias-corrected cumulative information reaches 1.928 bits (at least 1.8;
1.828 at 400 ms, 1.956 from the model rates), and the peak bias-corrected instantaneous information is 0.519
bits (within 0.5 +- 0.05; 0.540 from the model rates). The KDE matches the MATLAB tutorial's saved KDE to 0.0004
spikes/ms on average. With independent Monte Carlo draws per jackknife replicate the cumulative information still
peaks above 1.8 bits. The rate and onset neurons of Fig 1 are not in the repository, so the claim holds for one of
its three model neurons.

## Summary

| claim | outcome |
|---|---|
| units-404 | blocked (methods_underspecified): 914 and 404 units not identifiable; 1281 files labelled single |
| units-337, fig2a-poisson-test, fig2cd-fano-factor, fig2b-noise-correlations, s1-temporal-resolution, s2-spike-counts | blocked (methods_underspecified, from units-404) |
| fig6a-onset-sustained-information, fig6c-new-information, fig6b-fixed-rate-fraction, fig7a-time-constants, fig7b-k300, k300-rate-regression, fig9a-mean-cii, fig9a-top-quartile, s7-invariant-information, fig9b-cii-tau-ratio, fig9c-cii-k300, cii-unimodal | blocked (methods_underspecified, from units-337) |
| fig3-5-example-neurons | blocked (methods_underspecified): the three units are not named |
| table1-anatomy-counts, fig10-cii-distance, fig10-cii-area-anova, fig10-tau-k300-distance, fig10-no-anatomy-categorical | blocked (data_missing): no anatomical location in the data set |
| fig1-model-neurons | partially reproduced, on the one model neuron in the paper's repository |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

Differences from the paper's method: the KDE is a Python port of ssvkernel; the jackknife replicates of the Monte
Carlo estimate share their random draws (an assumption, with the independent variant shown); the bootstrap of the
jackknife (20 replicates) and the adaptive sample size of the Monte Carlo are not run, as the claim tested does not
use the error.

What was found about the data set: the files do not mark the 914 single auditory units or the 404 analysed units
(1281 files are labelled single units, 1246 pass the 2.3 bits/s auditory threshold); the anatomical locations and
coordinates of the recording sites are not in the data set (depths are in the file names); the numbers of stimuli
per unit match neither the paper nor the description; up to 12 trials per stimulus, against a stated maximum of 11;
three files store `sortType` as "singl"; the description names one bird WhiBlu5896M where the files say
WhiBlu5396M.